## 1) Importar bibliotecas

In [ ]:
from sklearn.model_selection import train_test_split, learning_curve, LearningCurveDisplay
from sklearn.metrics import PredictionErrorDisplay
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt

import polars as pl
import numpy as np

## 2) Ler base de dados

In [ ]:
data = pl.read_parquet(
    source = "./diabetes_dataset.parquet"
)

print(data.shape)
data.head(2)

## 3) Treinamento do modelo

### 3.1) Separar por `train_test_split`:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data.drop("target"),
    data["target"],
    random_state = 1,
    train_size = 0.7,
)

for array in [X_train, X_test, y_train, y_test]:
    print(array.shape)

### 3.2) Treinar modelo:

In [ ]:
model = LinearRegression()

model.fit(
    X = X_train,
    y = y_train
)

## 4) Avaliar modelo:

#### 4.1) `PredictionErrorDisplay`:

Modelos de Regressão Linear podem ser avaliados com uma nova métrica da biblioteca `sklearn` chamado `PredictionErrorDisplay`.

Basicamente, ele avalia se o modelo linear está errando acima ou abaixo do esperado.

In [ ]:
display = PredictionErrorDisplay(
    y_true = y_test,
    y_pred = model.predict(
        X = X_test
    )
)

display.plot()
plt.title("Performance do modelo LinearRegression (Sem pré-processamento):")
plt.show()

### 4.2) `learning_curve`:

Para avaliar a evolução do aprendizado conforme tamanho do dataset de treino, podemos usar o `learning_curve` para ver até onde o modelo começa a operar conforme esperado.

In [ ]:
learning_curve_results = learning_curve(
    estimator = model,
    X = data.drop("target"),
    y = data["target"],
    shuffle = True,
    random_state = 1,
    train_sizes = np.linspace(start = 0.01, stop = 1, num = 10)
)

print(learning_curve_results)

In [ ]:
display_learning_curve = LearningCurveDisplay(
    train_sizes = learning_curve_results[0],
    train_scores = learning_curve_results[1],
    test_scores = learning_curve_results[2],
    score_name = "R2"
)

display_learning_curve.plot()

## 5) Criar Pipeline

### 5.1) `ColumnTransformer`

No caso, o método `ColumnTransformer` garante que o `Pipeline` - que é na verdade um sequenciamento de etapas pré-determinadas pelo programador - opere com os dados processados em escala e em formato ideal.

In [ ]:
column_transformer = ColumnTransformer(
    transformers = [
        ("OneHotEnc", OneHotEncoder(drop = "if_binary"), ["sex"]),
        ("StandartScaler", StandardScaler(), ["age", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"])
    ]
)

pipeline = Pipeline(
    steps = [
        ("preprocessing", column_transformer),
        ("linear_model", LinearRegression())
    ]
)

pipeline.fit(
    X = X_train, y = y_train
)

fig, axs = plt.subplots(
    ncols = 2,
    figsize = (12, 5)
)

display1 = PredictionErrorDisplay(
    y_true = y_test,
    y_pred = pipeline.predict(
        X = X_test
    )
)

display2 = PredictionErrorDisplay(
    y_true = y_test,
    y_pred = model.predict(
        X = X_test
    )
)

display1.plot(ax = axs[0])
display2.plot(ax = axs[1])

axs[0].set_title("Com pré-processamento:")
axs[1].set_title("Sem pré-processamento:")

plt.suptitle("LinearModel não garante mudança absurda mesmo com Pipeline e ColumnTransformer", fontsize = 16)
plt.show()

In [ ]:
pipeline["linear_model"].coef_

In [ ]:
model.coef_